# 08 · Scaling out on a GPU

Notebook 06 read a real catalog and dispersed a modest 1st-order field on a CPU. This one is the **throughput** companion: a dense field with **all three spectral orders**, sized by a single knob so it runs small on a laptop and *large on a GPU*. It is deliberately **GPU-first** — the default batch is small enough to run anywhere, and the point is what happens when you turn the knob up on a GPU node.

Two simplifications make it a clean scaling test:

- **Sources are placed directly in SCA pixel coordinates** (random, and allowed to spill off the edges), so there is no sky→focal-plane step and the focus stays on the batched dispersion itself. (Sky placement works fine anywhere on the sky — notebook 07 — it is just not what we are timing.)
- **Spectra are drawn (with repetition) from the bundled templates**, so they are physical, while the **density is arbitrary** — we are not limited by the catalog's 1/100 sub-sample (notebook 06).

## 0 · Setup — all three orders

In [ ]:
import os
from pathlib import Path
os.environ.setdefault("JAX_COMPILATION_CACHE_DIR", str(Path.home() / ".cache" / "roman_grs_jax"))

import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import AsinhNorm
import astropy.units as u
import synphot as syn
import jax
import jax.numpy as jnp

from roman_disperser import paths, refdata, psf_model, star_disperser, galaxy_disperser, sersic, pipeline
from roman_disperser.elements import GRISM
from roman_disperser.optical_model import RomanOpticalModel
import roman_disperser.optical_model_jax as omj
import tutorial_helpers as th

SCA = 5
element = GRISM
ORDERS = list(element.orders)                     # ("0", "1", "2") for the grism
model = RomanOpticalModel(config_file=str(paths.optical_model_path(element=element)))
wl_um, wl_a, dlam_a = th.wavelength_grid(element)
wl_j = jnp.asarray(wl_um)

# per-order payloads, PSFs, sensitivities, dispersers, and batched fori
# loops — built once. The element maps each order to its STPSF filter
# (order 2 shares the order-1 PSF: stpsf_filters["2"] == "GRISM1").
opt = {o: omj.make_sca_payload(model, sca=SCA, order=o) for o in ORDERS}
psf = {o: psf_model.get_or_make_psf_payload(detector=f"WFI{SCA:02d}", order=o, element=element,
            cache_dir=str(paths.psf_cache_dir()), verbose=False) for o in ORDERS}
sens = pipeline.load_sensitivities(paths.sensitivity_dir(element=element), SCA, wl_um, orders=ORDERS)
star_fori = {o: pipeline.make_batched_star_fori(
                star_disperser.make_star_disperser(psf[o], opt[o]), sens[o], wl_j, dlam_a) for o in ORDERS}
gal_fori = {o: pipeline.make_batched_galaxy_fori(
                galaxy_disperser.make_galaxy_disperser(psf[o], opt[o]), sens[o], wl_j, dlam_a) for o in ORDERS}
OV = psf["1"]["oversample"]
print("JAX backend:", jax.default_backend(), "| devices:", jax.devices())

## 1 · A random field, sized by a knob

Set `N_STARS` / `N_GAL`. The defaults are small so this runs in a few minutes on a laptop CPU; on a GPU node bump them by ~100× for a realistic dense field. Sources are scattered over `[-500, 4588]` in both axes — a margin beyond the 4088-pixel detector — so some traces enter from off-chip, just like a real exposure.

We assemble FLAM spectra (the batched disperser applies `× sensitivity × Δλ` internally): one G0V template for stars, scaled by a random magnitude; a handful of redshifted Kinney–Calzetti templates for galaxies (z ≥ 1.2 so they cover the band), each galaxy also getting a random Sérsic stamp.

In [ ]:
N_STARS, N_GAL = 60, 15          # laptop defaults; try 5000 / 2000 on a GPU
rng = np.random.default_rng(0)
LO, HI = -500.0, 4588.0
f158 = refdata.get_f158_band()

# stars: one G0V FLAM template, scaled by random magnitude
star_flam0 = refdata.get_template("g0v").normalize(18 * u.ABmag, band=f158)(
    wl_a * u.AA, flux_unit=syn.units.FLAM).value
star_x = rng.uniform(LO, HI, N_STARS)
star_y = rng.uniform(LO, HI, N_STARS)
star_flam = (star_flam0[None, :] * 10 ** (-0.4 * (rng.uniform(16, 21, N_STARS) - 18))[:, None]).astype(np.float32)

# galaxies: a bank of redshifted KC96 FLAM templates, sampled with repetition
bank = []
for tmpl in ("kc96_starb1", "kc96_elliptical"):
    for z in (1.2, 1.6, 2.0, 2.5):
        sp = syn.SourceSpectrum(refdata.get_template(tmpl).model, z=z).normalize(20 * u.ABmag, band=f158)
        bank.append(sp(wl_a * u.AA, flux_unit=syn.units.FLAM).value)
bank = np.array(bank, np.float32)

gal_x = rng.uniform(LO, HI, N_GAL)
gal_y = rng.uniform(LO, HI, N_GAL)
gal_flam = (bank[rng.integers(0, len(bank), N_GAL)]
            * 10 ** (-0.4 * (rng.uniform(20, 22, N_GAL) - 20))[:, None]).astype(np.float32)
reff = sersic.catalog_r_eff_to_pixels(rng.uniform(0.2, 0.5, N_GAL), 0.11, OV)
gal_img = np.asarray(sersic.make_sersic_images(reff, rng.uniform(0.7, 4.0, N_GAL),
                                               rng.uniform(0.3, 1.0, N_GAL),
                                               rng.uniform(0, np.pi, N_GAL), 30 * OV))
gal_img = gal_img / gal_img.sum(axis=(1, 2), keepdims=True)
print(f"{N_STARS} stars + {N_GAL} galaxies, scattered over [{LO:.0f}, {HI:.0f}] px")

## 2 · Disperse all three orders

For each order we run the batched star loop and the batched galaxy loop, accumulating into one detector image. Every source × order is a single `fori_loop` iteration; the compile happens on the first call and is reused.

In [ ]:
field = jnp.zeros((4088, 4088), jnp.float32)
t = time.time()
for o in ORDERS:
    field = pipeline.disperse_batched_stars(star_fori[o], star_flam, star_x, star_y, field, batch_size=1000)
    field = pipeline.disperse_batched_galaxies(gal_fori[o], gal_flam, gal_x, gal_y,
                                               jnp.asarray(gal_img), field, batch_size=256)
field.block_until_ready()
elapsed = time.time() - t
field = np.asarray(field)

n_op = (N_STARS + N_GAL) * len(ORDERS)
print(f"{N_STARS} stars + {N_GAL} galaxies × {len(ORDERS)} orders = {n_op} dispersions "
      f"in {elapsed:.1f} s on {jax.default_backend()} — first pass, pays the JIT compile "
      f"({1e3*elapsed/n_op:.0f} ms each)")

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(field, origin="lower", cmap="inferno",
          norm=AsinhNorm(linear_width=field.max()*0.001, vmin=0, vmax=field.max()))
ax.set(title=f"random field on SCA{SCA}: {N_STARS} stars + {N_GAL} galaxies, all 3 orders",
       xlabel="x [pix]", ylabel="y [pix]")
fig.tight_layout()

## 3 · Compile vs steady state

The pass above paid the one-time costs: JIT-compiling six batched loops (3 orders × star and galaxy) plus each kernel's first-execution warm-up. At these small defaults that dominates the wall clock on a GPU, so its per-source rate says almost nothing about throughput. Production runs see the *steady-state* rate, so the cell below re-times the identical pass with everything already compiled and projects from that. (For corroboration: microbenchmarks of this deposit on an A10G measure ~10 ms per galaxy-order and ~3–4 ms per star-order at steady state.)

On a CPU the compile is a minor share and the two rates nearly coincide — the contrast is the point on a GPU. The same cells run unchanged on a GPU node: open this notebook with the GPU kernel (or `pixi run -e gpu`), set `N_STARS`/`N_GAL` to realistic numbers (thousands), and re-run.

In [ ]:
# identical pass, compiles already paid: this is the steady-state rate
field2 = jnp.zeros((4088, 4088), jnp.float32)
t = time.time()
for o in ORDERS:
    field2 = pipeline.disperse_batched_stars(star_fori[o], star_flam, star_x, star_y, field2, batch_size=1000)
    field2 = pipeline.disperse_batched_galaxies(gal_fori[o], gal_flam, gal_x, gal_y,
                                                jnp.asarray(gal_img), field2, batch_size=256)
field2.block_until_ready()
steady = time.time() - t
del field2

ms = 1e3 * steady / n_op
print(f"first pass (compile + warm-up): {1e3*elapsed/n_op:7.0f} ms per source-order")
print(f"steady state                  : {ms:7.1f} ms per source-order on {jax.default_backend()}")
for n in (1000, 10000):
    full = n * 3                      # all three orders
    print(f"{n:>6,} sources (×3 orders) ≈ {ms/1e3*full:6.0f} s on this backend at the steady rate")
print("\nProduction runs (notebook 06's full pipeline) are GPU-only: the one-time compile")
print("amortizes away at scale, so the steady rate is the number that matters.")

## Recap

- Placing sources in **SCA pixels** (random, off-edge allowed) gives a dense field with no sky→focal-plane step — the simplest way to keep the timing about dispersion alone.
- **All three grism orders** come straight from the element (`element.orders`, with the order→PSF-filter mapping in `element.stpsf_filters`), batched exactly as in notebook 06 and co-added.
- The work is **dispersion-bound and embarrassingly parallel**, so it scales linearly in sources once the one-time compile amortizes — set the knob and re-run on a GPU.

**Next — [09 · The prism](09_prism.ipynb).** WFI's second dispersing element: the same pipeline with `element=PRISM`, and what actually changes — trace, dispersion, resolving power.